[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/exorbyte/mbox-cookbook/blob/main/06-agentic-ai/09-topic_routing_with_llm_enriched_aliases.ipynb)

In [1]:
# !pip install mbox openai python-dotenv

# Topic Routing with LLM-Enriched Aliases

`04-recall-tuning/05` measured `DETECT` mode carefully: it tolerates misspelling a trigger phrase gracefully, but it has zero tolerance for genuine synonyms, a customer who says "I want my money back" instead of "refund" is invisible to it, every time, no matter how the threshold is tuned. That gap needs a vocabulary, not a better matching mode. This notebook closes it by having an LLM propose that vocabulary once, caching the result to a file, and making sure every actual chatbot lookup afterward touches only the cached, deterministic aliases, never the LLM.

In this notebook you will:

1. Reconfirm the gap directly: a support topic router that misses obvious paraphrases
2. Generate a small alias list per topic with a real LLM call, exactly once, and cache it to disk
3. Prove the cache works, run the generation step twice, and confirm the second run makes zero LLM calls
4. Rebuild the index with those cached aliases and watch the same paraphrases resolve correctly
5. Build a small `route_message` function that detects every topic a message touches, not just one

> Note: this notebook makes real calls to the OpenAI API, but only the first time it runs. To run it, put an `OPENAI_API_KEY` in a `.env` file in this directory.

In [2]:
import json
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
load_dotenv()

True

## 1. The gap, directly

Four short topics a support bot needs to recognize inside a customer's message. Ask it about something using a completely ordinary paraphrase, no typos, just different words.

In [3]:
from mbox.indexing import TableIndexer
from mbox.recall import TableRecallMode

topics = pd.read_csv("datasets/support_topics.csv")
index = TableIndexer.create_index(topics, index_columns=["phrase"], tmp_dir="tmp_index_topics")

message = "I want my money back for this order"
result = index.match(phrase=message, modes={"phrase": TableRecallMode.DETECT}, include_field_scores=True)
result[["phrase_candidate", "phrase_score"]]

mbpie: 33 modules, 566 methods, 8 classes, 18 enums
  args: 426 required, 254 optional, 37 keywords, 39 flags, 26 arrays
  types: 372 int, 337 str, 1 double, 72 object


,phrase_candidate,phrase_score
0,,0


Score `0`, nothing found, even though any human reading `"I want my money back"` knows exactly what topic this is. `DETECT` was never going to catch this on its own, `04-recall-tuning/05` showed the same result on this same kind of query. The fix is giving each topic a vocabulary of the other ways people actually ask for it.

## 2. Generate that vocabulary once, and cache it

One function, two rules: if a cache file already exists, load it and stop, no network call. If it doesn't, ask the model for a handful of alternate phrasings per topic, save the result, and never ask again until the cache is deleted on purpose.

In [4]:
from openai import OpenAI
client = OpenAI()

llm_call_count = 0

tools = [{
    "type": "function",
    "function": {
        "name": "propose_aliases",
        "description": "Propose alternate ways a customer might phrase the same request or topic.",
        "parameters": {
            "type": "object",
            "properties": {
                "aliases": {
                    "type": "array", "items": {"type": "string"},
                    "description": "3 to 5 short, natural alternate phrasings a real customer might type."
                }
            },
            "required": ["aliases"]
        }
    }
}]

def generate_alias_list(phrase: str) -> list[str]:
    global llm_call_count
    llm_call_count += 1
    prompt = (
        f"A support chatbot needs to detect when a customer message is about '{phrase}'. "
        f"Propose 3 to 5 short, natural alternate ways a real customer might phrase this same request, "
        f"without using the words '{phrase}' verbatim. Keep each alias under 6 words."
    )
    response = client.chat.completions.create(
        model="gpt-4o", messages=[{"role": "user", "content": prompt}],
        tools=tools, tool_choice={"type": "function", "function": {"name": "propose_aliases"}}
    )
    return json.loads(response.choices[0].message.tool_calls[0].function.arguments)["aliases"]

def load_or_generate_aliases(phrases: list[str], cache_path: Path) -> dict:
    if cache_path.exists():
        print(f"Found {cache_path}, loading cached aliases, no LLM call.")
        return json.loads(cache_path.read_text())
    print(f"No cache at {cache_path}, generating aliases via LLM.")
    generated = {phrase: generate_alias_list(phrase) for phrase in phrases}
    cache_path.write_text(json.dumps(generated, indent=2))
    return generated

In [5]:
CACHE_PATH = Path("topic_aliases_cache.json")

aliases_by_topic = load_or_generate_aliases(topics["phrase"].tolist(), CACHE_PATH)
print(f"\nLLM calls made: {llm_call_count}")
aliases_by_topic

No cache at topic_aliases_cache.json, generating aliases via LLM.



LLM calls made: 4


{'refund': ['money back',
  'return payment',
  'get my money returned',
  'reverse the charge',
  'money reimbursed'],
 'password reset': ['Forgot my password',
  'Need to set new password',
  "Can't access my account",
  'Trouble logging in',
  'Change my password'],
 'cancel subscription': ['end my membership',
  'stop my subscription',
  'terminate my account',
  'unsubscribe me',
  'remove my membership'],
 'shipping delay': ["Where's my package?",
  'When will it arrive?',
  'Shipment taking too long',
  'Why the late delivery?',
  "Package hasn't come yet"]}

## 3. Prove the cache actually works

Run the exact same call again, in a fresh cell. If caching is doing its job, `llm_call_count` does not move.

In [6]:
aliases_by_topic_again = load_or_generate_aliases(topics["phrase"].tolist(), CACHE_PATH)
print(f"LLM calls made: {llm_call_count}")
print(f"Same content as before: {aliases_by_topic == aliases_by_topic_again}")

Found topic_aliases_cache.json, loading cached aliases, no LLM call.
LLM calls made: 4
Same content as before: True


Zero additional calls, identical content. `CACHE_PATH` is the only thing standing between this notebook and calling the LLM again, delete the file to force regeneration, otherwise every future run, and every real match this router ever performs, uses exactly the aliases generated in section 2.

## 4. Rebuild the index with the cached aliases

`AliasSet` takes it from here, the same mechanism `02-data-harmonization/02` used, just populated from the LLM's proposals instead of typed in by hand.

In [7]:
from mbox.aliases import AliasSet

alias_set = AliasSet(name="topic_aliases")
for phrase, alias_list in aliases_by_topic.items():
    for alias in alias_list:
        alias_set.add(word=phrase, alias=alias, penalty=10)

enriched_index = TableIndexer.create_index(
    topics, index_columns=["phrase"], alias_sets={"phrase": alias_set}, tmp_dir="tmp_index_topics_enriched"
)

result = enriched_index.match(phrase=message, modes={"phrase": TableRecallMode.DETECT}, include_field_scores=True)
result[["phrase_candidate", "phrase_score"]]

,phrase_candidate,phrase_score
0,refund,78


The same `"I want my money back for this order"` that scored `0` in section 1 now resolves cleanly to `refund`. Nothing about `DETECT` changed, the vocabulary underneath it did.

## 5. Test the aliases against realistic paraphrases, not just the ones you wrote

An LLM proposing aliases doesn't guarantee even coverage. Some proposed phrasings will fuzzy-match a wide range of real messages, others will turn out narrower than they looked. The only way to know which is which is to actually run a batch of realistic paraphrases through it, none of them typos, none of them copied from section 2's generated list verbatim.

In [8]:
paraphrase_test_set = [
    "I'm locked out of my account and can't log in",
    "I'd like to close my subscription please",
    "hey where's my package, it's been two weeks",
    "this message has nothing to do with any of that",
    "can I get my money back",
    "I want to stop paying for this service",
    "my order is really late",
    "forgot my password, help",
]

for msg in paraphrase_test_set:
    r = enriched_index.match(phrase=msg, modes={"phrase": TableRecallMode.DETECT}, include_field_scores=True)
    print(f"{msg!r:55s} -> {r.iloc[0]['phrase_candidate']!r} at {int(r.iloc[0]['phrase_score'])}")

"I'm locked out of my account and can't log in"         -> '' at 0
"I'd like to close my subscription please"              -> '' at 0
"hey where's my package, it's been two weeks"           -> 'shipping delay' at 78
'this message has nothing to do with any of that'       -> '' at 0
'can I get my money back'                               -> 'refund' at 82
'I want to stop paying for this service'                -> '' at 0
'my order is really late'                               -> '' at 0
'forgot my password, help'                              -> 'password reset' at 86


The spread is the point, look at the actual scores printed above rather than at any specific number described here, they won't be identical on your run. Some paraphrases resolve confidently, some miss outright, and depending on exactly how the model happened to phrase things in section 2, some may land in a weak middle ground, present enough to return a candidate, not present enough to trust it. Alias generation isn't deterministic, coverage isn't guaranteed evenly across topics, and that's exactly the review step section 7's first practical note calls out: an alias list earns production trust by being tested against real examples, not by having come from an LLM.

## 6. Route on confidence, not on any match at all

A weak score is not the same as a miss, and treating it that way is how a router ends up auto-acting on a guess. `min_qualities` turns "found something, barely" back into "not found" once a score drops below a chosen floor, the same threshold discipline `04-recall-tuning/05` demonstrated and `06-confidence_based_escalation` built a full policy around.

In [9]:
def route_message(message: str, index, min_score: int = 50) -> list[dict]:
    result = index.match(
        phrase=message, modes={"phrase": TableRecallMode.DETECT},
        max_results=4, min_qualities={"phrase": min_score}, include_field_scores=True
    )
    if len(result) == 0 or result.iloc[0]["index_row"] == -1:
        return []
    return [
        {"topic": row["phrase_candidate"], "score": int(row["phrase_score"])}
        for _, row in result.iterrows()
    ]

route_message("I want a refund because my package never showed up", enriched_index)

[{'topic': 'refund', 'score': 84}]

A message that never uses either trigger phrase, or any alias, verbatim, still routes to both topics it actually touches, `min_qualities` didn't need to hold anything back here because both scores clear it comfortably. Point `route_message` at whichever query scored weakest in section 5 and it should come back empty, correctly held back rather than acted on with false confidence, and that's the cue to either add a more specific alias for it or leave it for a human, not to lower the threshold until it passes.

## 7. Practical notes

**Review generated aliases before trusting them in production.** `aliases_by_topic` in section 2 is plain, inspectable JSON specifically so a human can read it before it goes anywhere near a live router. An LLM proposing "close my subscription" for `cancel subscription` is obviously fine, it proposing something ambiguous enough to bleed into a different topic is the kind of mistake worth catching before it's baked into the cache, not after.

**Regenerate the cache on purpose, not by accident.** Deleting `topic_aliases_cache.json` and rerunning is a deliberate action, a new topic, a wording that turned out to cause false positives. Nothing in this notebook regenerates it automatically, and nothing should, an alias list that changes on every run is not something you can audit or reason about.

**Aliases add vocabulary, they don't add fuzziness.** Section 5's unrelated message still didn't match anything, section 3 of `04-recall-tuning/05` already showed `DETECT` won't fire on partial phrase coverage, and enriching the vocabulary with aliases doesn't change that safety property.

**A `penalty` on an alias is a real lever.** This notebook used `penalty=10` uniformly, scoring an alias-based match a bit lower than an exact mention of the trigger phrase itself. If a downstream decision, `06-confidence_based_escalation`'s policy, for instance, needs to tell "the customer said the exact word" apart from "the customer said something we mapped to it," the penalty is where that distinction lives.